# Exploración inicial de los logs

Esta notebook representa el primer análisis manual antes de automatizar el reporte.

Objetivos:

1. Confirmar qué archivos existen y su tamaño.
2. Ver cómo se agrupan los eventos mediante `operation_Id`.
3. Contar los reseteos que ADManager confirmó como exitosos.

Los logs permanecen en `data/raw/`, no se guardan salidas en esta notebook y no se publican en Git.

In [ ]:
from collections import Counter
from pathlib import Path
import re

RAIZ_PROYECTO = Path.cwd().parent
DIRECTORIO_LOGS = RAIZ_PROYECTO / 'data' / 'raw'
archivos_log = sorted(DIRECTORIO_LOGS.glob('*.log'))
PATRON_OPERACION = re.compile(r'operation_Id=([^\]]+)')

[(archivo.name, archivo.stat().st_size) for archivo in archivos_log]

## Eventos y operaciones

Un archivo tiene muchas líneas, pero varias pertenecen a la misma operación.
Aquí comprobamos la diferencia usando el primer log disponible.

In [ ]:
archivo = archivos_log[0]
lineas = archivo.read_text(encoding='utf-8').splitlines()
operation_ids = [coincidencia.group(1) for linea in lineas
                 if (coincidencia := PATRON_OPERACION.search(linea))]
operaciones = Counter(operation_ids)

print(f'Archivo: {archivo.name}')
print(f'Líneas leídas: {len(lineas)}')
print(f'Operaciones encontradas: {len(operaciones)}')

operation_id, cantidad_eventos = operaciones.most_common(1)[0]
print(f'Eventos en una operación: {cantidad_eventos}')

## Reseteos confirmados

La regla no usa únicamente HTTP 200. Solo acepta una operación cuando la
respuesta de ADManager confirma `reset: yes`, el mensaje de éxito y estado `1`.

In [ ]:
MARCAS_DE_EXITO = (
    "'reset': 'yes'",
    "'statusMessage': 'Password reset successful.'",
    "'status': '1'",
)
reseteos_exitosos = 0

for archivo in archivos_log:
    for linea in archivo.read_text(encoding='utf-8').splitlines():
        if all(marca in linea for marca in MARCAS_DE_EXITO):
            reseteos_exitosos += 1

print(f'Reseteos exitosos: {reseteos_exitosos}')
print('Este hallazgo definirá la regla del programa.')